# Tutorial

This tutorial demonstrates how to use LineageVI for RNA velocity analysis with gene programs.


In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import lineagevi
import scanpy as sc
import scvelo as scv
import numpy as np
import os

In [ ]:
adata = scv.datasets.gastrulation_erythroid()

In [ ]:
#adata = sc.read_h5ad('/Users/lgolinelli/git/lineageVI/docs/data/Pancreas/GSE132188_adata.h5ad.h5')

In [ ]:
adata.X = adata.layers['unspliced'].copy() + adata.layers['spliced'].copy()
adata.layers['counts'] = adata.X.copy()

In [ ]:
annotation_path = '/Users/lgolinelli/git/lineageVI/notebooks/data/inputs/gene_sets/'
annotation_name = 'msigdb_development_or_pancreas.gmt'
file_path = os.path.join(annotation_path, annotation_name)
lineagevi.utils.add_annotations(
    adata, 
    files=[file_path],
    min_genes=12,
    varm_key='I',
    uns_key='terms',
    clean=True,
    genes_use_upper=True)

adata._inplace_subset_var(adata.varm['I'].sum(1) > 0)

In [ ]:
scv.pp.filter_and_normalize(adata, min_shared_counts=20, n_top_genes=2000, subset_highly_variable=True, log=True)

#sc.pp.highly_variable_genes(adata, n_top_genes=2000, subset=True, flavor='seurat_v3')

Filter out any annotations (terms) with less than 12 genes.

In [ ]:
select_terms = adata.varm['I'].sum(0)>12
adata.uns['terms'] = np.array(adata.uns['terms'])[select_terms].tolist()
adata.varm['I'] = adata.varm['I'][:, select_terms]

Filter out genes not present in any retained terms after selection of HVGs.

In [ ]:
adata._inplace_subset_var(adata.varm['I'].sum(1)>0)

In [ ]:
scv.pp.moments(adata, n_pcs=100, n_neighbors=200)
sc.tl.leiden(adata)

In [ ]:
lineagevi.utils.compute_nearest_neighbors(adata, K=20, neighbors_key='neighbors', indices_key='indices')  # Gets 20 neighbors + self = 21 total

In [ ]:
vae = lineagevi.LineageVI(
    adata=adata,
    n_hidden=128,
    mask_key='I',
    unspliced_key='Mu',
    spliced_key='Ms',
    nn_key='indices',
    cluster_key='leiden',  # Key in adata.obs
    cluster_embedding_dim=32,  # Optional, default is 32
    cls_embedding_dim=8,
    cls_encoding_key = 'Ciao',
)

history = vae.fit(
    K=10,
    batch_size=256,
    lr=1e-3,
    epochs1=50,
    epochs2=50,
    seeds=(0, 1, 2),
    output_dir='/Users/lgolinelli/git/lineageVI/notebooks/data/outputs/pancreas',   # or None
    verbose=1,
    monitor_genes=['Gnas', 'Rbfox3'],
    monitor_negative_velo=True,
    monitor_every_epochs=5
)


In [ ]:
vae.get_model_outputs(
    adata=adata,
    return_negative_velo=True,
    base_seed=0,
    save_to_adata=True,
    unspliced_key='Mu',
    spliced_key='Ms',
    nn_key='indices',
    rescale_velocity_magnitude=True, # the more turbulent the velocity flow, the smaller the velocity magnitude
    max_velocity_magnitude=1
)


In [ ]:
adata_gp = lineagevi.utils.build_gp_adata(adata)

In [ ]:
lineagevi.plots.plot_phase_plane(adata,  'Tram1', cluster_key='leiden', u_scale=.1, s_scale=.1, alpha=1, head_width=0.02, head_length=0.03, length_includes_head=False)
lineagevi.plots.plot_phase_plane(adata,  'Jph1', cluster_key='leiden', u_scale=.1, s_scale=.1, alpha=1, head_width=0.02, head_length=0.03, length_includes_head=False)
lineagevi.plots.plot_phase_plane(adata,  'Kcnq5', cluster_key='leiden', u_scale=.1, s_scale=.1, alpha=1, head_width=0.02, head_length=0.03, length_includes_head=False)
lineagevi.plots.plot_gp_phase_planes(
    adata_gp,
    program_pairs=[('MEGAKARYOCYTE_MARKERS', 'FACTORS_INVOLVED_IN_MEGAKARYOC')],
    cluster_key='leiden',
    title='Phase Planes',
    figsize_per_panel = (7, 7),
    alpha = 1,
    arrow_multiplier=.5
)

In [ ]:
df_normalized = vae.compute_cluster_alignment_matrix(normalize=True, cluster_key='leiden', aggregation='weighted_mean')
fig, ax = lineagevi.plots.plot_cluster_alignment_matrix(df_normalized)

In [ ]:
# Simple usage
similarity_df = lineagevi.utils.compute_cluster_embedding_similarity(adata, cluster_names='leiden')
# Plot it
fig, ax = lineagevi.plots.plot_cluster_alignment_matrix(similarity_df,)

In [ ]:
# Standalone usage: Switch cluster labels for cells
# This modifies adata in place, so use a copy if needed
#adata_copy = adata.copy()  
# Example: Switch cells from 'Alpha' cluster to 'Beta' cluster
df_genes, df_gps = vae.perturb_cluster_labels(
    adata,
    source_cluster='1',
    target_cluster='7'
)


In [ ]:
adata_gp = lineagevi.utils.build_gp_adata(adata)

In [ ]:
adata.obs

In [ ]:
lineagevi.plots.top_features_table(adata_gp, groupby_key="celltype", categories=['Erythroid3'], layer="velocity", n=5)

In [ ]:
# Evaluate how perturbing GP activations affects alignment
results_gp = vae.evaluate_perturbation_effects(
    adata,
    perturbation_type='gps',
    groupby_key='celltype',
    group_to_perturb='Alpha',
    gps_to_perturb=['FETAL_KIDNEY_ERYTHROBLASTS', 'FACTORS_INVOLVED_IN_MEGAKARYOC'],  # Gene program names from adata.uns['terms']
    perturb_value=1.0,  # Replace GP activation with this value
    gp_uns_key='terms',  # default 'terms'
    use_gp_space=False,
    normalize=True,
    aggregation='weighted_mean',
    compute_embedding_similarity=False,  # Cluster embeddings are unchanged for GP perturbations
)

# Visualize results
import lineagevi.plots as lv_plots

# Alignment change
fig, ax = lv_plots.plot_cluster_alignment_matrix(
    results_gp['alignment_delta'],
    title='Alignment Change After GP Perturbation'
)

# Check summary statistics
print(f"Max increase: {results_gp['alignment_delta'].max().max():.3f}")
print(f"Max decrease: {results_gp['alignment_delta'].min().min():.3f}")
print(f"Mean absolute change: {results_gp['alignment_delta'].abs().mean().mean():.3f}")

In [ ]:
# Evaluate how perturbing gene expression affects alignment
results_gene = vae.evaluate_perturbation_effects(
    adata,
    perturbation_type='genes',
    groupby_key='clusters',
    group_to_perturb='Alpha',
    genes_to_perturb=['Gnas', 'Gcg'],  # Gene names from adata.var_names
    perturb_value=0,  # Add this value to expression
    use_gp_space=False,
    normalize=True,
    aggregation='weighted_mean',
    compute_embedding_similarity=False,  # Cluster embeddings are unchanged for gene perturbations
)

# Visualize results
fig, ax = lv_plots.plot_cluster_alignment_matrix(
    results_gene['alignment_delta'],
    title='Alignment Change After Gene Perturbation'
)


In [ ]:
sc.pp.neighbors(adata)
sc.tl.umap(adata)
sc.pl.umap(adata, color=['clusters', 'leiden'])

In [ ]:
results_labels['alignment_delta']

In [ ]:
# Evaluate how switching cluster labels affects alignment and embedding similarity
results_labels = vae.evaluate_perturbation_effects(
    adata,
    perturbation_type='cluster_labels',
    groupby_key='clusters',
    source_cluster='3',  # Original cluster
    target_cluster='2',   # Target cluster
    use_gp_space=False,
    normalize=True,
    aggregation='weighted_mean',
    compute_embedding_similarity=True,  # Important: cluster embeddings change here!
)

# Visualize alignment changes
fig, ax = lv_plots.plot_cluster_alignment_matrix(
    results_labels['alignment_delta'],
    title='Alignment Change After Cluster Label Switch'
)

# Visualize embedding similarity changes (only meaningful for cluster label perturbations)
if results_labels['similarity_delta'] is not None:
    fig, ax = lv_plots.plot_cluster_alignment_matrix(
        results_labels['similarity_delta'],
        title='Embedding Similarity Change After Cluster Label Switch'
    )
    
    # Check summary statistics for similarity
    print(f"Max similarity increase: {results_labels['similarity_delta'].max().max():.3f}")
    print(f"Max similarity decrease: {results_labels['similarity_delta'].min().min():.3f}")
    print(f"Mean absolute similarity change: {results_labels['similarity_delta'].abs().mean().mean():.3f}")

# Check summary statistics for alignment
print(f"Max alignment increase: {results_labels['alignment_delta'].max().max():.3f}")
print(f"Max alignment decrease: {results_labels['alignment_delta'].min().min():.3f}")
print(f"Mean absolute alignment change: {results_labels['alignment_delta'].abs().mean().mean():.3f}")

In [ ]:
import lineagevi.plots as lv_plots

# Evaluate perturbation effects
results = vae.evaluate_perturbation_effects(
    adata,
    perturbation_type='gps',
    group_to_perturb='Alpha',
    gps_to_perturb=['REGULATION_OF_BETA_CELL_DEVELO'],  # Example GP names
    perturb_value=0,  # Increase GP expression by 1.0
    use_gp_space=True,  # Compute alignment in GP space
    normalize=True,
    aggregation='weighted_mean',
    compute_embedding_similarity=True,
)

# Visualize baseline alignment
fig, ax = lv_plots.plot_cluster_alignment_matrix(
    results['baseline_alignment'],
    title='Baseline Alignment'
)

# Visualize perturbed alignment
fig, ax = lv_plots.plot_cluster_alignment_matrix(
    results['perturbed_alignment'],
    title='Alignment After Perturbation'
)

# Visualize alignment changes (delta)
fig, ax = lv_plots.plot_cluster_alignment_matrix(
    results['alignment_delta'],
    title='Alignment Change (Perturbed - Baseline)',
    center=0  # Center colormap at 0
)

# Visualize baseline similarity
fig, ax = lv_plots.plot_cluster_alignment_matrix(
    results['baseline_similarity'],
    title='Baseline Embedding Similarity'
)

# Visualize similarity changes
fig, ax = lv_plots.plot_cluster_alignment_matrix(
    results['similarity_delta'],
    title='Similarity Change (Perturbed - Baseline)',
    center=0  # Center colormap at 0
)

# Print summary statistics
print("\n=== Alignment Change Summary ===")
print(f"Max increase: {results['alignment_delta'].max().max():.3f}")
print(f"Max decrease: {results['alignment_delta'].min().min():.3f}")
print(f"Mean absolute change: {results['alignment_delta'].abs().mean().mean():.3f}")

print("\n=== Similarity Change Summary ===")
print(f"Max increase: {results['similarity_delta'].max().max():.3f}")
print(f"Max decrease: {results['similarity_delta'].min().min():.3f}")
print(f"Mean absolute change: {results['similarity_delta'].abs().mean().mean():.3f}")

In [ ]:
sc.pp.neighbors(adata)
#sc.tl.umap(adata)
scv.tl.velocity_graph(adata, vkey='velocity')
scv.pl.velocity_embedding_stream(adata, color='celltype', vkey='velocity')

In [ ]:
adata_gp = lineagevi.utils.build_gp_adata(adata)

In [ ]:
sc.pp.neighbors(adata_gp)
sc.tl.umap(adata_gp)
scv.tl.velocity_graph(adata_gp)
scv.pl.velocity_embedding_stream(adata_gp, color='celltype')

In [ ]:
'''vae.map_velocities(
        adata,
        adata_gp=None,
        direction = "gp_to_gene",
        scale = 10.0,
        velocity_key = "velocity_gp2gene",
        unspliced_key = "Mu",
        spliced_key = "Ms"
)
'''

In [ ]:
'''vae.map_velocities(
        adata,
        adata_gp=adata_gp,
        direction = "gene_to_gp",
        scale = 10.0,
        velocity_key = "velocity_gene2gp",
        unspliced_key = "Mu",
        spliced_key = "Ms"
)'''

In [ ]:
lineagevi.plots.top_features_table(adata, groupby_key="clusters", categories=['Beta'], layer="velocity", n=10)

In [ ]:
lineagevi.plots.top_features_table(adata_gp, groupby_key="clusters", categories=['Beta'], layer="velocity", n=20)

In [ ]:
lineagevi.plots.top_features_table(adata_gp, groupby_key="clusters", categories=['Epsilon'], layer="velocity", n=20)

In [ ]:
lineagevi.plots.plot_gp_phase_planes(
    adata_gp,
    program_pairs=[('REGULATION_OF_BETA_CELL_DEVELO', 'WT_VS_TRAF6KO_DAY6_EFF_CD8_TCE')],
    cluster_key='clusters',
    title='Phase Planes',
    figsize_per_panel = (7, 7),
    alpha = 1,
    arrow_multiplier=.5
)

In [ ]:
sc.pl.violin(adata, keys='Gcg', layer='alpha', groupby='clusters')

In [ ]:
vae.model.latent_enrich(adata, groups='clusters', comparison='rest', n_sample=5000, key_added='bf_scores')

In [ ]:
lineagevi.plots.plot_abs_bfs(adata, scores_key='bf_scores', n_cols=4, n_points=10, lim_val=2.3, fontsize=8, scale_y=2, yt_step=0.3,
                    title=None, figsize=None, dpi=400)

In [ ]:
sc.pl.scatter(adata_gp, x='REGULATION_OF_BETA_CELL_DEVELO',y='SHH_UP_LATE.V1_UP', color='clusters', layers='velocity', size=10, alpha=0.5, title='Clusters')

In [ ]:
df_genes, df_gps, perturbed_outputs = vae.perturb_genes(
                adata=adata,
                groupby_key='clusters',
                group_to_perturb='Beta',
                genes_to_perturb=['Sntg1', 'Snhg6'],
                perturb_value=0,
                perturb_spliced=True,
                perturb_unspliced=True,
            )


In [ ]:
for key, value in perturbed_outputs.items():
    print(key)
    print('---')

In [ ]:
df_genes, df_gps, perturbed_outputs = vae.perturb_gps(
                adata=adata,
                gp_uns_key='terms',
                gps_to_perturb=['YBX1_TARGETS_DN', 'YBX1_TARGETS_UP'],
                groupby_key='clusters',
                group_to_perturb='Beta',
                perturb_value=0,
            )

In [ ]:
df_genes

In [ ]:
df_gps

In [ ]:
for key, value in perturbed_outputs.items():
    print(key)
    print('---')


In [ ]:
'''vae.get_directional_uncertainty(
    adata,
    use_gp_velo = False,
    n_samples = 50,
    n_jobs = -1,
    show_plot =  True,
    base_seed = None,
)

vae.get_directional_uncertainty(
    adata,
    use_gp_velo = False,
    n_samples = 50,
    n_jobs = -1,
    show_plot =  True,
    base_seed = None,
)

df = vae.compute_extrinsic_uncertainty(
    adata,
    use_gp_velo=True,
    n_samples=25, 
    n_jobs=-1,
    show_plot=True)


df = vae.compute_extrinsic_uncertainty(
    adata,
    use_gp_velo=True,
    n_samples=25, 
    n_jobs=-1,
    show_plot=True) '''

In [ ]:
# Single cluster pair alignment (gene expression space)
alignment = vae.compute_cluster_alignment(
    source_cluster='Alpha',
    target_cluster='Beta'
)

# Single cluster pair alignment (GP space)
alignment_gp = vae.compute_cluster_alignment(
    source_cluster='Alpha',
    target_cluster='Beta',
    use_gp_space=True
)

# Full alignment matrix (gene expression space)
matrix, clusters = vae.compute_cluster_alignment_matrix()

# Full alignment matrix with weighted mean aggregation
matrix_weighted, clusters = vae.compute_cluster_alignment_matrix(
    aggregation='weighted_mean'
)

# GP space alignment matrix
matrix_gp, clusters = vae.compute_cluster_alignment_matrix(
    use_gp_space=True
)

In [ ]:
# Compute alignment matrix - returns DataFrame with cluster names as indices
df = vae.compute_cluster_alignment_matrix()

# Access specific alignment using cluster names
alignment_ab = df.loc['Alpha', 'Beta']

# Find which cluster Alpha points toward
target_cluster = df.loc['Alpha'].idxmax()
alignment = df.loc['Alpha', target_cluster]
print(f"Alpha → {target_cluster}: {alignment:.3f}")

# Visualize directly with seaborn (no need to specify labels!)
import seaborn as sns
import matplotlib.pyplot as plt
sns.heatmap(df, cmap='RdBu_r', center=0, annot=True, fmt='.2f')
plt.xlabel('Target Cluster')
plt.ylabel('Source Cluster')
plt.title('Cluster Alignment Matrix')
plt.show()

# Filter or subset easily
# All clusters that Alpha points toward (sorted by alignment)
alpha_targets = df.loc['Alpha'].sort_values(ascending=False)

# All clusters that point toward Beta
sources_to_beta = df.loc[:, 'Beta'].sort_values(ascending=False)

In [ ]:
# This will now work correctly
df = vae.compute_cluster_alignment_matrix()
alignment_ab = df.loc['Alpha', 'Beta']  # ✅ Works!